In [1]:
from Montreal_UHI_toolbox import obs, obs_rural, obs_urban
from UHI_statistics import load_daily_simobs,avail_thresh,alpha
import xarray as xr
import numpy as np
from scipy import stats

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [25]:
# Given 2 urban stations, 7 rural stations, create bootstraped random samples of rural stations to estimate error
n = len(obs_rural.station) # 7 samples
n_boot = 5000 # Number of rural bootstrap station selections
err_btsrp = {} # Saves 95% confidence errors indexed by [field=tasmin/tasmax/tas][model=Station/CLASS/TEB+CLASS][season=JJA/SON/DJF/MAM] 

for f in ['tasmin','tasmax','tas']:
    err_btsrp[f] = {}

    # Load rural station/model obs/simobs
    rural_S = obs_rural.where(obs_rural.count(dim='station') >= len(obs_rural.station)*avail_thresh,drop=True)[f] # Load Station obs - tweak to UHI_statistics rural_S
    rural_C = load_daily_simobs(field=f,model='C').sel(station=obs_rural.station.values) # Load CLASS rural simobs
    rural_T = load_daily_simobs(field=f,model='T').sel(station=obs_rural.station.values) # Load TEB+CLASS rural simobs

    # Create many sampling sets of n = 7 rural stations with stations likely repeating
    for station_sample in np.random.choice(obs_rural.station,size=(n_boot, len(obs_rural.station)),replace=True):
        
        # Some kind of indexing standard
        for m in ['C','T','S']:
            err_btsrp[f][m] = {}

        # For each bootstrapped sample, solve standard 95% confidence errors:

        for s in ['JJA','SON','DJF','MAM']:  # First grab the mean of each of the sampled rural stations
            err_btsrp[f]['S'][s] = rural_S.sel(station=station_sample).groupby('time.season')[s].mean(dim='time') # Stations
            err_btsrp[f]['C'][s] = rural_C.sel(station=station_sample).groupby('time.season')[s].mean(dim='time') # CLASS
            err_btsrp[f]['T'][s] = rural_T.sel(station=station_sample).groupby('time.season')[s].mean(dim='time') # TEB+CLASS

            for m in ['C','T','S']: 
                # Find the standard deviation between the n = 7 bootstrap sampled station averages
                err_btsrp[f][m][s] = err_btsrp[f][m][s].std(dim='station')

                # Scale this standard deviation to 95% standard error based on n = 7 samples
                err_btsrp[f][m][s] *= stats.t.ppf(1 - alpha/2, n-1)/np.sqrt(n)     

In [26]:

for f in ['tasmin','tasmax','tas']:
    print('----------------')
    for s in ['JJA','SON','DJF','MAM']:
        print()
        for m in ['C','T','S']:
            print(f'{f}_{m}_{s} ERR ±{err_btsrp[f][m][s].values}C')

----------------

tasmin_C_JJA ERR ±0.2247082013603856C
tasmin_T_JJA ERR ±0.17569897445316993C
tasmin_S_JJA ERR ±0.39986774317823803C

tasmin_C_SON ERR ±0.19939021573877763C
tasmin_T_SON ERR ±0.1658959838396854C
tasmin_S_SON ERR ±0.3899463122967669C

tasmin_C_DJF ERR ±0.43674487323796685C
tasmin_T_DJF ERR ±0.3739698448359863C
tasmin_S_DJF ERR ±0.5162391381358379C

tasmin_C_MAM ERR ±0.306725442930294C
tasmin_T_MAM ERR ±0.2644669251375462C
tasmin_S_MAM ERR ±0.32418770603249564C
----------------

tasmax_C_JJA ERR ±0.273468313056324C
tasmax_T_JJA ERR ±0.2509702579695003C
tasmax_S_JJA ERR ±0.2311965165656349C

tasmax_C_SON ERR ±0.3891966985614243C
tasmax_T_SON ERR ±0.37479176350431453C
tasmax_S_SON ERR ±0.429355269694486C

tasmax_C_DJF ERR ±0.5390964031939072C
tasmax_T_DJF ERR ±0.5370318868671746C
tasmax_S_DJF ERR ±0.4598500417220382C

tasmax_C_MAM ERR ±0.4271934375175262C
tasmax_T_MAM ERR ±0.41699824586760265C
tasmax_S_MAM ERR ±0.2838105920932523C
----------------

tas_C_JJA ERR ±0.2815103